In [ ]:
import gc
import shutil
import tempfile
from pathlib import Path

import lightgbm as lgb
import numpy as np
import polars as pl
from sklearn.preprocessing import TargetEncoder

# Configure Polars print outs
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)

In [ ]:
RAW_INPUT_DATA_DIR = Path("/kaggle/input/competitions/m5-forecasting-accuracy")
FEATURES_DATA_PATH = Path("/kaggle/input/datasets/lmalms/m5-acc-2-0/m5-acc-features-df.pq")
SUBMISSION_DIR = Path("/kaggle/working/")

# Per-store predictions are written here and combined into the final submission at the end.
PREDICTIONS_DIR = Path(tempfile.mkdtemp(prefix="store_predictions_"))
print(f"Store predictions directory: {PREDICTIONS_DIR}")

In [ ]:
# Data constants
LABEL_COL_NAME = "sales"

# Only the most recent MAX_TRAIN_DAYS days are used for training + validation.
# Lower this to reduce RAM usage (None = use everything).
MAX_TRAIN_DAYS: int | None = None
MAX_TRAIN_INDEX = 1941
MIN_TRAIN_INDEX = 0 if MAX_TRAIN_DAYS is None else MAX_TRAIN_INDEX - MAX_TRAIN_DAYS

VALIDATION_FORECAST_HORIZON = 0
VALID_START_INDEX = MAX_TRAIN_INDEX - VALIDATION_FORECAST_HORIZON + 1
print(
    f"Train period: [{MIN_TRAIN_INDEX}, {VALID_START_INDEX}); "
    f"{VALID_START_INDEX - MIN_TRAIN_INDEX - 1} days"
)
print(
    f"Valid period: [{VALID_START_INDEX} - {max(MAX_TRAIN_INDEX, VALID_START_INDEX)}); "
    f"{max(MAX_TRAIN_INDEX, VALID_START_INDEX) - VALID_START_INDEX} days"
)

SUBMISSION_FORECAST_HORIZON = 28

# Feature engineering constants. These must match what the feature engineering
# notebook (1.1) used to build the features parquet.
SALES_LAGS = [1, 2, 3, 4, 7, 14]
ROLLING_SALES_WINDOWS = [2, 3, 4, 7, 14]
ROLLING_SALES_STATS = ["mean", "std", "min", "max"]
PRICE_LAGS = [1, 2, 3, 4, 7]
ROLLING_PRICE_WINDOWS = [2, 3, 4, 7, 14]

CYCLICAL_CALENDAR_FEATURE_SUFFIXES = ["wday", "month"]

N_EVENTS = 2
MAX_DAYS_UNTIL_EVENT = 7
MAX_DAYS_SINCE_EVENT = 7

# Election (SNAP) features are state specific: each store only gets the ones of its own state.
SNAP_STATES = ["CA", "TX", "WI"]
MAX_DAYS_UNTIL_ELECTION = 7
MAX_DAYS_SINCE_ELECTION = 7

### Feature configuration

In [ ]:
# Feature configuration
# Feature groups are built per state, as only the election features of a store's own state are used.
def build_feature_groups(state_id: str) -> dict[str, list[str]]:
    election_cols = [f"snap_{state_id}"]
    return {
        # store_id / state_id are constant within a store so are not included.
        "id_categorical": ["item_id", "dept_id", "cat_id"],

        # Sales features
        "sales_lags": [f"sales_lagged_{lag}" for lag in SALES_LAGS],
        **{
            f"sales_rolling_{stat}": [f"sales_rolling_{stat}_{w}" for w in ROLLING_SALES_WINDOWS]
            for stat in ROLLING_SALES_STATS
        },

        # Price features
        "sell_price": ["sell_price"],
        "price_lags": [f"sell_price_lagged_{lag}" for lag in PRICE_LAGS],
        "price_rolling_mean": [f"sell_price_rolling_mean_{w}" for w in ROLLING_PRICE_WINDOWS],
        "price_rolling_std": [f"sell_price_rolling_std_{w}" for w in ROLLING_PRICE_WINDOWS],

        # Calendar features
        "calendar_cyclical": [
            f"{trig}_{suffix}"
            for suffix in CYCLICAL_CALENDAR_FEATURE_SUFFIXES
            for trig in ("sin", "cos")
        ],
        "calendar_raw": ["wday", "month", "year"],
        "is_weekend": ["is_weekend"],

        # Event features
        "event_categorical": [
            f"{prefix}_{i}"
            for prefix in ("event_name", "event_type", "last_event_type", "next_event_type")
            for i in range(1, N_EVENTS + 1)
        ],
        "event_timing": [
            f"days_{direction}_event_{i}"
            for direction in ("since", "until")
            for i in range(1, N_EVENTS + 1)
        ],

        # Election features (state specific)
        "snap": list(election_cols),
        "election_timing": [
            f"days_{direction}_{e}"
            for direction in ("since", "until")
            for e in election_cols
        ],
    }


CATEGORICAL_FEATURE_GROUPS = {"id_categorical", "event_categorical"}
TARGET_ENCODER_FEATURE_COLS = {"item_id"}

# ---- Configure features ----
# Add / remove groups to include / drop all of its features from training and inference.
FEATURE_GROUPS_TO_INCLUDE = [
    "id_categorical",
    "event_categorical",
    "sell_price",
    "sales_lags",
    "sales_rolling_mean",
    "sales_rolling_std",
    "sales_rolling_min",
    "sales_rolling_max",
    "price_lags",
    "price_rolling_mean",
    "price_rolling_std",
    "calendar_cyclical",
    # "calendar_raw",
    "is_weekend",
    "event_timing",
    "snap",
    "election_timing",
]

# Add / remove individual features
EXTRA_FEATURES_TO_INCLUDE: list[str] = []
FEATURES_TO_EXCLUDE: list[str] = []
# ------------------------


def get_feature_cols(state_id: str) -> tuple[list[str], list[str]]:
    """Returns (all feature columns, categorical feature columns) for stores in the given state."""
    groups = build_feature_groups(state_id)
    categorical_candidates = {c for g in CATEGORICAL_FEATURE_GROUPS for c in groups[g]}
    feature_cols = list(
        dict.fromkeys(
            c
            for c in [*(c for g in FEATURE_GROUPS_TO_INCLUDE for c in groups[g]), *EXTRA_FEATURES_TO_INCLUDE]
            if c not in FEATURES_TO_EXCLUDE
        )
    )
    categorical_cols = [c for c in feature_cols if c in categorical_candidates]
    return feature_cols, categorical_cols


for _state in SNAP_STATES:
    _feature_cols, _categorical_cols = get_feature_cols(_state)
    print(f"{_state}: {len(_feature_cols)} features ({len(_categorical_cols)} categorical)")

### Model configuration

In [ ]:
# LightGBM config
NUM_BOOST_ROUND = 5000
EARLY_STOPPING_ROUNDS = 10

LGB_PARAMS = {
    # "objective": "poisson",
    # "metric": ["poisson", "rmse"],
    "objective": "tweedie",
    "tweedie_variance_power": 1.2,
    "metric": ["tweedie", "rmse"],
    "learning_rate": 2e-03,         
    "num_leaves": 63,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "max_bin": 255,
    "cat_smooth": 20,
    "force_row_wise": True,        
    "seed": 0,
    "num_threads": -1,
    "verbosity": -1,
}

### Shared inference data

Calendar, event and election features are identical across stores, so they are computed once for all states. Each store only picks up the election columns of its own state.

In [ ]:
# Load data needed for inference

# Calendar data
calendar_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    **{date_type: pl.Int16 for date_type in ["wday", "month", "year"]},
    **{f"snap_{state}": pl.Int16 for state in SNAP_STATES},
}
raw_calendar_df = pl.read_csv(
    RAW_INPUT_DATA_DIR / "calendar.csv",
    try_parse_dates=True,
    schema_overrides=calendar_schema_overrides,
)

# Sell price data
sell_price_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    "sell_price": pl.Float32
}
raw_sell_price_df = pl.read_csv(
    RAW_INPUT_DATA_DIR / "sell_prices.csv",
    schema_overrides=sell_price_schema_overrides,
)

# Submission
sample_submission_df = pl.read_csv(RAW_INPUT_DATA_DIR / "sample_submission.csv")

In [ ]:
# Calendar / event / election features are recomputed for inference.
# Logic must match what the feature engineering notebook (1.1) used to build the features parquet.

event_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "event_name_1", "event_type_1", "event_name_2", "event_type_2"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        pl.when(pl.col(f"event_name_{i}").is_not_null())
        .then(pl.col("d_index"))
        .otherwise(None)
        .alias(f"event_index_{i}")
        for i in range(1, N_EVENTS + 1)
    )
    .with_columns(
        *[
            (pl.col("d_index") - pl.col(f"event_index_{i}").forward_fill())
            .alias(f"days_since_event_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.col(f"event_type_{i}").forward_fill().alias(f"last_event_type_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            (pl.col(f"event_index_{i}").backward_fill() - pl.col("d_index"))
            .alias(f"days_until_event_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.col(f"event_type_{i}").backward_fill().alias(f"next_event_type_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
    )
    .with_columns(
        # Clip days since / until events (and the associated event types)
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= MAX_DAYS_SINCE_EVENT)
            .then(pl.col(f"days_since_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= MAX_DAYS_SINCE_EVENT)
            .then(pl.col(f"last_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_until_event_{i}") <= MAX_DAYS_UNTIL_EVENT)
            .then(pl.col(f"days_until_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_until_event_{i}") <= MAX_DAYS_UNTIL_EVENT)
            .then(pl.col(f"next_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
    )
    .drop([f"event_index_{i}" for i in range(1, N_EVENTS + 1)])
)

event_features_df.head()

In [ ]:
# Calendar / event / election features are recomputed for inference.
# Logic must match what the feature engineering notebook (1.1) used to build the features parquet.

calendar_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "wday", "month", "year"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        sin_wday=(2 * np.pi * pl.col("wday") / 7).sin().cast(pl.Float32),
        cos_wday=(2 * np.pi * pl.col("wday") / 7).cos().cast(pl.Float32),
        sin_month=(2 * np.pi * pl.col("month") / 12).sin().cast(pl.Float32),
        cos_month=(2 * np.pi * pl.col("month") / 12).cos().cast(pl.Float32),
        is_weekend=(pl.col("wday") <= 2).cast(pl.Float32),
    )
)

calendar_features_df.head()

In [ ]:
# Calendar / event / election features are recomputed for inference.
# Logic must match what the feature engineering notebook (1.1) used to build the features parquet.

election_features_df = (
    raw_calendar_df
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        pl.when(pl.col(election).eq(1))
        .then(pl.col("d_index"))
        .otherwise(None)
        .alias(f"{election}_index")
        for election in ELECTION_COLS
    )
    .select(["wm_yr_wk", "date", "d", "d_index"] + ELECTION_COLS + [f"{e}_index" for e in ELECTION_COLS])
    .with_columns(
        *[
            (pl.col("d_index") - pl.col(f"{e}_index").forward_fill()).alias(f"days_since_{e}")
            for e in ELECTION_COLS
        ],
        *[
            (pl.col(f"{e}_index").backward_fill() - pl.col("d_index")).alias(f"days_until_{e}")
            for e in ELECTION_COLS
        ],
    )
    .with_columns(
        *[
            pl.when(pl.col(f"days_since_{e}") <= MAX_DAYS_SINCE_ELECTION)
            .then(pl.col(f"days_since_{e}"))
            .otherwise(None)
            for e in ELECTION_COLS
        ],
        *[
            pl.when(pl.col(f"days_until_{e}") <= MAX_DAYS_UNTIL_ELECTION)
            .then(pl.col(f"days_until_{e}"))
            .otherwise(None)
            for e in ELECTION_COLS
        ],
    )
    .drop([f"{e}_index" for e in ELECTION_COLS])
)

election_features_df.head()

In [ ]:
# Merge calendar / events / election features once
# and keep only the forecast days (small).

forecast_calendar_df = (
    calendar_features_df.drop("d", "date")
    .join(event_features_df.drop("d", "date"), on=["wm_yr_wk", "d_index"], how="inner", validate="1:1")
    .join(election_features_df.drop("d", "date"), on=["wm_yr_wk", "d_index"], how="inner", validate="1:1")
    .filter(pl.col("d_index") > MAX_TRAIN_INDEX)
)
forecast_calendar_df.head()

In [ ]:
# Forecast identifiers: parse each submission id into its hierarchy columns once.
SUBMISSION_F_COLS = [f"F{i}" for i in range(1, SUBMISSION_FORECAST_HORIZON + 1)]

_SERIES_ID_RE = (
    r"^(?P<cat>[A-Z]+)_(?P<dept>\d+)_(?P<item>\d+)_(?P<state>[A-Z]+)_(?P<store>\d+)"
    r"_(?:validation|evaluation)$"
)

forecast_ids_df = (
    sample_submission_df
    .filter(pl.col("id").str.ends_with("_evaluation"))
    .select("id")
    .unique()
    .with_columns(parsed_id=pl.col("id").str.extract_groups(_SERIES_ID_RE))
    .with_columns(
        item_id=pl.format(
            "{}_{}_{}",
            pl.col("parsed_id").struct.field("cat"),
            pl.col("parsed_id").struct.field("dept"),
            pl.col("parsed_id").struct.field("item"),
        ),
        dept_id=pl.format(
            "{}_{}",
            pl.col("parsed_id").struct.field("cat"),
            pl.col("parsed_id").struct.field("dept"),
        ),
        cat_id=pl.col("parsed_id").struct.field("cat"),
        store_id=pl.format(
            "{}_{}",
            pl.col("parsed_id").struct.field("state"),
            pl.col("parsed_id").struct.field("store"),
        ),
        state_id=pl.col("parsed_id").struct.field("state"),
    )
    .drop("parsed_id")
)

STORE_IDS = sorted(forecast_ids_df["store_id"].unique().to_list())
print(STORE_IDS)

## Store level model

In [ ]:
DATA_COLS_BASE = ["id", "d_index", LABEL_COL_NAME]
max_lookback = max(SALES_LAGS + ROLLING_SALES_WINDOWS + PRICE_LAGS + ROLLING_PRICE_WINDOWS)


def load_store_data(store_id: str, feature_cols: list[str]) -> pl.DataFrame:
    """Training data for a single store (only the features of its state)."""
    data_cols = list(dict.fromkeys([*DATA_COLS_BASE, *feature_cols]))
    return (
        pl.scan_parquet(FEATURES_DATA_PATH)
        .filter(pl.col("store_id") == store_id)
        .filter(pl.col("d_index") > MIN_TRAIN_INDEX)
        .select(data_cols)
        .with_columns(pl.col(pl.Float64).cast(pl.Float32))
        .collect()
    )


def load_store_history(store_id: str) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Final slice of sales / price history of a store needed to build features for the first forecast day."""
    history_df = (
        pl.scan_parquet(FEATURES_DATA_PATH)
        .filter(pl.col("store_id") == store_id)
        .filter(pl.col("d_index") > MAX_TRAIN_INDEX - max_lookback)
        .select(["id", "d_index", "sales", "sell_price"])
        .with_columns(pl.col("sell_price").cast(pl.Float32))
        .sort(["id", "d_index"])
        .collect()
    )
    return history_df.select(["id", "d_index", "sales"]), history_df.select(["id", "d_index", "sell_price"])


def to_float32_matrix(
    df: pl.DataFrame,
    feature_cols: list[str],
    category_dtypes: dict[str, pl.Enum],
    target_encoder: TargetEncoder,
    is_train: bool,
) -> np.ndarray:
    """
    Feature matrix (float32, NaN = missing) in feature_cols order.
    Categoricals become integer codes (LightGBM treats them as categories through `categorical_feature`).
    """
    X = np.empty((df.height, len(feature_cols)), dtype=np.float32)

    # Prepare target encodings
    target_encoder_cols = [c for c in feature_cols if c in TARGET_ENCODER_FEATURE_COLS]
    if target_encoder_cols:
        if is_train:
            X_target_encoded = target_encoder.fit_transform(
                df[target_encoder_cols].to_numpy(),
                df[LABEL_COL_NAME].to_numpy().astype(np.float32),
            )
        else:
            X_target_encoded = target_encoder.transform(df[target_encoder_cols].to_numpy())

    for j, col in enumerate(feature_cols):
        s = df.get_column(col)
        if col in category_dtypes:
            s = s.cast(pl.String).cast(category_dtypes[col], strict=False).to_physical()
            v = s.cast(pl.Float32).to_numpy()
        elif col in target_encoder_cols:
            v = X_target_encoded[:, target_encoder_cols.index(col)]
        else:
            v = s.cast(pl.Float32).to_numpy()
        X[:, j] = v
    return X

In [ ]:
def train_store_model(store_id: str, state_id: str) -> dict:
    """Trains a LightGBM model on the data of a single store."""
    feature_cols, categorical_cols = get_feature_cols(state_id)
    full_df = load_store_data(store_id, feature_cols)
    print(f"[{store_id}] data: {full_df.shape}, {full_df.estimated_size('gb'):.2f} GB")

    target_encoder = TargetEncoder(smooth="auto", target_type="continuous")

    # Construct category mapping from training data only for any categories
    # that will not be target encoded.
    category_dtypes = {
        col: pl.Enum(
            full_df.filter(pl.col("d_index") < VALID_START_INDEX)[col]
            .cast(pl.String).drop_nulls().unique().sort()
        )
        for col in categorical_cols
        if col not in TARGET_ENCODER_FEATURE_COLS
    }

    def matrix(df: pl.DataFrame, is_train: bool) -> np.ndarray:
        return to_float32_matrix(df, feature_cols, category_dtypes, target_encoder, is_train)

    # RAM: build one split at a time and delete the polars frames as soon as possible.
    is_valid = pl.col("d_index") >= VALID_START_INDEX

    train_df = full_df.filter(~is_valid)
    X_train = matrix(train_df, is_train=True)
    y_train = train_df[LABEL_COL_NAME].to_numpy().astype(np.float32)
    del train_df

    valid_df = full_df.filter(is_valid)
    X_valid: np.ndarray | None = None
    y_valid: np.ndarray | None = None
    if valid_df.height > 0:
        X_valid = matrix(valid_df, is_train=False)
        y_valid = valid_df[LABEL_COL_NAME].to_numpy().astype(np.float32)
    del valid_df, full_df
    gc.collect()

    train_set = lgb.Dataset(
        X_train,
        label=y_train,
        feature_name=feature_cols,
        categorical_feature=categorical_cols,
        params=LGB_PARAMS,
        free_raw_data=True,
    )
    valid_set: lgb.Dataset | None = None
    if X_valid is not None:
        valid_set = lgb.Dataset(
            X_valid,
            label=y_valid,
            reference=train_set,
            feature_name=feature_cols,
            categorical_feature=categorical_cols,
            params=LGB_PARAMS,
            free_raw_data=True,
        )

    # RAM: binning happens at construct(); afterwards the raw float32 matrices are not needed.
    train_set.construct()
    if valid_set is not None:
        valid_set.construct()
    del X_train, X_valid, y_train, y_valid
    gc.collect()

    model = lgb.train(
        LGB_PARAMS,
        train_set,
        num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[train_set] + ([valid_set] if valid_set is not None else []),
        valid_names=["train"] + (["valid"] if valid_set is not None else []),
        callbacks=(
            [lgb.log_evaluation(period=100)]
            + ([lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS)] if valid_set is not None else [])
        ),
    )
    if valid_set is not None:
        print(f"[{store_id}] best iteration: {model.best_iteration}, valid RMSE: {model.best_score['valid']['rmse']:.4f}")

    del train_set, valid_set
    gc.collect()

    return {
        "model": model,
        "feature_cols": feature_cols,
        "category_dtypes": category_dtypes,
        "target_encoder": target_encoder,
    }

In [ ]:
def predict_store(store_id: str, state_id: str, store_model: dict) -> pl.DataFrame:
    """Recursive 28 day forecast for the series of a single store. Returns long format (id, F, sales)."""
    model = store_model["model"]
    feature_cols = store_model["feature_cols"]

    # Only the election features of this store's state
    other_states = [s for s in SNAP_STATES if s != state_id]
    store_calendar_df = forecast_calendar_df.drop(
        [c for c in forecast_calendar_df.columns if any(c.endswith(f"snap_{s}") for s in other_states)]
    )

    store_forecast_ids_df = forecast_ids_df.filter(pl.col("store_id") == store_id)
    sales_history_df, price_history_df = load_store_history(store_id)

    # Recursive forecasting: iterate over forecast days, predict, and feed the predictions
    # back in as sales history for the features of the next day.
    all_forecast_dfs: list[pl.DataFrame] = []

    for f_index in range(1, SUBMISSION_FORECAST_HORIZON + 1):
        d_index = MAX_TRAIN_INDEX + f_index

        # Construct test_df with identifier columns
        test_df = (
            store_forecast_ids_df
            .with_columns(
                id=pl.col("id").cast(pl.Categorical),
                d_index=pl.lit(d_index).cast(pl.Int16),
                sales=pl.lit(None).cast(pl.Int16),
            )
            # Prices and calendar features for the current d_index are known in advance
            .join(
                other=store_calendar_df.select(["d_index", "wm_yr_wk"]),
                on="d_index", how="left", validate="m:1",
            )
            .join(
                other=raw_sell_price_df,
                on=["item_id", "store_id", "wm_yr_wk"], how="left", validate="m:1",
            )
            .drop("wm_yr_wk")
        )

        # Lagged / rolling sales features
        previous_sales_df = (
            pl.concat(
                [
                    sales_history_df,
                    *[
                        df.select(["id", "d_index", "sales"])
                        .with_columns(id=pl.col("id").cast(pl.Categorical))
                        for df in all_forecast_dfs
                    ],
                    test_df.select(["id", "d_index", "sales"]),
                ]
            )
            .sort(by=["id", "d_index"])
        )

        test_sales_features_df = (
            previous_sales_df
            .with_columns(
                [
                    pl.col("sales").shift(lag).over("id", order_by="d_index")
                    .alias(f"sales_lagged_{lag}")
                    for lag in SALES_LAGS
                ]
                + [
                    getattr(pl.col("sales").shift(1), f"rolling_{stat}")(window_size=window, min_samples=window)
                    .over("id", order_by="d_index")
                    .alias(f"sales_rolling_{stat}_{window}")
                    .cast(pl.Float32)
                    for stat in ROLLING_SALES_STATS
                    for window in ROLLING_SALES_WINDOWS
                ]
            )
            .drop("sales")
            .filter(pl.col("d_index") == d_index)
        )

        # Lagged / rolling price features
        price_history_df = (
            pl.concat([price_history_df, test_df.select(["id", "d_index", "sell_price"])])
            .sort(by=["id", "d_index"])
        )

        test_price_features_df = (
            price_history_df
            .with_columns(
                [
                    pl.col("sell_price").shift(lag).over("id", order_by="d_index")
                    .alias(f"sell_price_lagged_{lag}")
                    for lag in PRICE_LAGS
                ]
                + [
                    pl.col("sell_price").shift(1)
                    .rolling_mean(window_size=window, min_samples=window)
                    .over("id", order_by="d_index")
                    .alias(f"sell_price_rolling_mean_{window}")
                    .cast(pl.Float32)
                    for window in ROLLING_PRICE_WINDOWS
                ]
                + [
                    pl.col("sell_price").shift(1)
                    .rolling_std(window_size=window, min_samples=window)
                    .over("id", order_by="d_index")
                    .alias(f"sell_price_rolling_std_{window}")
                    .cast(pl.Float32)
                    for window in ROLLING_PRICE_WINDOWS
                ]
            )
            .drop("sell_price")
            .filter(pl.col("d_index") == d_index)
        )
        # Keep the price history bounded
        price_history_df = price_history_df.filter(pl.col("d_index") > d_index - max_lookback)

        # Join id, sales, price and calendar features
        test_df = (
            test_df.drop("sales")
            .join(test_sales_features_df, on=["id", "d_index"], how="inner", validate="1:1")
            .join(test_price_features_df, on=["id", "d_index"], how="inner", validate="1:1")
            .join(store_calendar_df, on="d_index", how="inner", validate="m:1")
            # Series without a price at this timestamp aren't sold: no forecast
            .filter(pl.col("sell_price").is_not_null())
        )

        # Make predictions. `predict` returns the expected count (inverse link is applied).
        # Use the best iteration found by early stopping (if validation was used).
        y_hat = model.predict(
            to_float32_matrix(
                test_df,
                feature_cols,
                store_model["category_dtypes"],
                store_model["target_encoder"],
                is_train=False,
            ),
            num_iteration=model.best_iteration if model.best_iteration > 0 else None,
        )

        # Sales can't be negative, and are rounded to integers so they can be fed back in
        # as lagged sales features.
        y_hat_df = (
            pl.DataFrame({"id": test_df["id"].cast(pl.String), "sales": y_hat})
            .with_columns(
                sales=pl.col("sales").clip(lower_bound=0).round().cast(pl.Int16),
                d_index=pl.lit(d_index).cast(pl.Int16),
                F=pl.format("F{}", f_index),
            )
        )
        all_forecast_dfs.append(y_hat_df)

    return pl.concat(all_forecast_dfs).select(["id", "F", "sales"])

## Train and predict for each store

In [ ]:
for store_id in STORE_IDS:
    state_id = store_id.split("_")[0]
    predictions_path = PREDICTIONS_DIR / f"{store_id}.pq"
    if predictions_path.exists():
        print(f"[{store_id}] predictions already exist, skipping")
        continue

    print(f"=== {store_id} ({state_id}) ===")
    store_model = train_store_model(store_id, state_id)

    # Feature importance
    print(
        pl.DataFrame({
            "feature": store_model["model"].feature_name(),
            "gain": store_model["model"].feature_importance(importance_type="gain"),
        })
        .sort("gain", descending=True)
        .with_columns(gain_share=pl.col("gain") / pl.col("gain").sum())
        .head(10)
    )

    predict_store(store_id, state_id, store_model).write_parquet(predictions_path)

    # RAM: free everything belonging to this store before moving on
    del store_model
    gc.collect()

## Submission

In [ ]:
final_forecast_df = (
    pl.read_parquet(PREDICTIONS_DIR / "*.pq")
    .with_columns(sales=pl.col("sales").cast(pl.Int64))
    .pivot(on="F", index="id", values="sales")
)

FINAL_SUBMISSION_DF = (
    sample_submission_df
    .join(final_forecast_df, on="id", how="left", suffix="_model", validate="1:1")
    .with_columns(
        pl.coalesce(pl.col(f"{c}_model"), pl.col(c)).cast(pl.Int64).alias(c)
        for c in SUBMISSION_F_COLS
    )
    .select(sample_submission_df.columns)
)

FINAL_SUBMISSION_DF.write_csv(SUBMISSION_DIR / "submission.csv")
print(FINAL_SUBMISSION_DF.shape, f"{final_forecast_df.height} / {len(forecast_ids_df)} series forecast by model")

shutil.rmtree(PREDICTIONS_DIR, ignore_errors=True)